# instruments
twelve widgets, each one built around a question a domain expert already asks: which files are old, which site needs a visit, where the glitches are in a long recording. four of them come in pairs, where the second widget takes its inputs from the first.


In [4]:
import subprocess, numpy as np, pandas as pd, vibe_widget as vw
vw.config(model="google/gemini-3.8-flash", bypass_row_guard=True)   # the 200k-sample trace below
rng = np.random.default_rng(3)

### code review · where is the old code that still gets touched
real data: `git blame` of this repo. the question a maintainer has before a refactor.

In [5]:
def blame(path):
    out = subprocess.run(["git", "-C", "..", "blame", "--line-porcelain", "--", path], capture_output=True, text=True).stdout.splitlines()
    rows, author, when = [], None, None
    for l in out:
        if l.startswith("author "): author = l[7:]
        elif l.startswith("author-time "): when = int(l[12:])
        elif l.startswith("\t"): rows.append((path.split("src/vibe_widget/")[1], len(rows) + 1, len(l) - 1, author, when))
    return rows
files = subprocess.run(["git", "-C", "..", "ls-files", "src/vibe_widget/*.py", "src/vibe_widget/**/*.py"], capture_output=True, text=True).stdout.split()
lines = pd.DataFrame([r for f in files for r in blame(f)], columns=["file", "line_no", "line_len", "author", "t"])
lines["age_days"] = ((pd.Timestamp.now().timestamp() - lines.t) / 86400).round().astype(int)
churn = subprocess.run(["git", "-C", "..", "log", "--since=90.days", "--name-only", "--format="], capture_output=True, text=True).stdout.split()
lines["churn_90d"] = lines.file.map(pd.Series(churn).str.split("src/vibe_widget/").str[-1].value_counts()).fillna(0).astype(int)
lines = lines.drop(columns="t")
print(len(lines), "lines in", lines.file.nunique(), "files"); lines.sample(3, random_state=0)

12180 lines in 38 files


,file,line_no,line_len,author,age_days,churn_90d
2288,core/widget.py,1023,39,Dylan Wootton,258,5
9342,themes.py,334,20,Dylan,270,1
3730,core/widget.py,2465,29,Dylan,251,5


In [6]:
seesoft = vw.create(
    """seesoft view of a codebase. one thin column per file (sorted by path, file name written vertically at the top), inside it every line of code is a 1px-tall horizontal bar whose length is line_len (capped at the column width); no gaps.
    colour = age_days on a continuous scale from fresh (warm yellow) to old (deep blue), legend with the day range.
    a vertical slider on the right, 'younger than N days', draggable, starts at 30: lines older than N fade to light grey, the rest keep colour; badge 'k of n lines younger than N days · m files'. every drag step of the slider must recompute k and m from the data and redraw the canvas (the slider value lives in state, the canvas draw effect depends on it).
    hover a bar: file:line_no · author · age_days d · churn_90d commits. click a column to select that file (thick outline, output selected_file); click again to deselect. the canvas must handle 10k bars smoothly (draw on canvas).""",
    lines, theme="minimal",
    outputs=vw.outputs(selected_file="file path of the selected column or null", max_age="the slider value in days"),
)

### trial monitoring · which of 4000 sites needs a visit
synthetic. shaped like a site-performance table, no trial in it.

In [7]:
n = 4000
sites = pd.DataFrame({
    "site": [f"S{i:04d}" for i in range(n)],
    "country": rng.choice(["US", "DE", "JP", "BR", "IN", "PL", "CA", "AU"], n, p=[.3, .12, .1, .1, .12, .08, .1, .08]),
    "enrolled": rng.poisson(18, n),
    "screen_fail_pct": np.clip(rng.normal(22, 9, n), 0, 90).round(1),
    "deviations": rng.poisson(1.4, n),
    "open_queries": rng.poisson(6, n),
    "query_age_days": np.clip(rng.exponential(9, n), 0, 120).round(),
    "dropout_pct": np.clip(rng.normal(8, 5, n), 0, 60).round(1),
    "days_since_visit": np.clip(rng.normal(70, 30, n), 5, 400).round(),
})
bad = rng.choice(n, 40, replace=False)   # planted: sites that are quietly falling apart
sites.loc[bad, ["deviations", "open_queries", "query_age_days", "days_since_visit"]] *= [4, 5, 6, 3]
sites.head()

,site,country,enrolled,screen_fail_pct,deviations,open_queries,query_age_days,dropout_pct,days_since_visit
0,S0000,US,18,5.9,2,7,14.0,15.3,14.0
1,S0001,US,13,17.3,3,7,47.0,11.4,113.0
2,S0002,PL,24,19.3,2,0,10.0,6.8,98.0
3,S0003,BR,18,26.8,0,6,4.0,7.7,5.0
4,S0004,US,16,20.1,1,5,1.0,8.5,103.0


In [8]:
lens = vw.create(
    """table lens for 4000 rows. one 1px row per record, the 7 numeric columns (enrolled, screen_fail_pct, deviations, open_queries, query_age_days, dropout_pct, days_since_visit) each drawn as a horizontal bar in its own column track, bar length = value / column max; country as a thin colour strip on the left.
    click a column header to sort all rows by it descending, click again for ascending, an arrow shows the sort.
    drag on the left row axis to set a focus range: rows inside expand to a readable height (~18px) and show the text values (site, country, numbers), rows outside stay 1px; drag the focus body to slide it, drag its edges to resize, up/down arrow keys move it one row. start with a 12-row focus at the top.
    hover any 1px row to see its site and values in a tooltip. draw the compressed rows on canvas.""",
    sites, theme="minimal",
    outputs=vw.outputs(focus_rows="list of site ids inside the focus range", sort_col="column currently sorted by"),
)

### long recording · find the glitches in 200k samples
synthetic 1 kHz signal, 200 s, seven spikes and one dropout planted.

In [9]:
t = np.arange(200_000) / 1000
v = np.sin(2 * np.pi * 0.7 * t) + 0.3 * np.sin(2 * np.pi * 13 * t) + rng.normal(0, 0.05, t.size)
for s in rng.choice(t.size, 7, replace=False): v[s:s + 3] += rng.choice([-1, 1]) * rng.uniform(3, 6)
v[151_200:151_900] = 0                                   # dropout
sig = pd.DataFrame({"v": v.astype("float32")})
sig.shape

(200000, 1)

In [10]:
trace = vw.create(
    """overview + detail for a long signal: column v, 200000 samples at 1000 Hz (time = index / 1000 s).
    top strip (full width, ~90px): the min/max envelope of the whole recording per pixel column, drawn on canvas. a window on it, starts at 0–2 s: drag its body to move, drag its edges to resize, left/right arrow keys step by one window width, shift+arrows by a tenth.
    bottom (~300px): the raw samples inside the window, every sample, drawn on canvas, y axis fitted to the window's data, time axis in seconds; it must be painted on first mount (size the canvas from its container in a layout effect, then draw), not only after the first interaction.
    double-click in the detail drops a numbered event marker at that time; markers persist (also as ticks on the overview), each listed on the right with time and the local peak value; click a list entry to jump the window there; Delete removes the highlighted marker.""",
    sig, theme="minimal",
    outputs=vw.outputs(window="[t0, t1] in seconds", events="list of marked times in seconds"),
)

### attention heads · which ones do previous-token or induction
synthetic attention for 6 layers × 8 heads over a repeated sentence, with a previous-token head, an induction head and a couple of BOS sinks planted. the pair: grid picks a head, the detail follows.

In [11]:
tokens = ["<s>", "the", "cat", "sat", "on", "the", "mat", ".", "the", "cat", "sat", "on", "the", "mat", "."]
L, H, n = 6, 8, len(tokens)
def head(kind):
    w = np.zeros((n, n))
    for q in range(n):
        if kind == "prev" and q > 0: w[q, q - 1] = 1
        elif kind == "bos": w[q, 0] = 1
        elif kind == "induction":
            prev = [k for k in range(q) if tokens[k] == tokens[q]]
            w[q, prev[-1] + 1 if prev and prev[-1] + 1 <= q else 0] = 1
        else: w[q, :q + 1] = rng.dirichlet(np.ones(q + 1) * 0.6)
        w[q] = 0.85 * w[q] / w[q].sum() + 0.15 * (np.arange(n) <= q) / (q + 1)
    return w
kinds = {(1, 3): "prev", (4, 2): "induction", (0, 5): "bos", (2, 0): "bos"}
attn = pd.DataFrame([(l, h, q, k, float(head(kinds.get((l, h), "mixed"))[q, k]))
                     for l in range(L) for h in range(H) for q in range(n) for k in range(n)], columns=["layer", "head", "q", "k", "w"])
attn = attn[attn.w > 1e-4].reset_index(drop=True)
toks = pd.DataFrame({"pos": range(n), "token": tokens})
len(attn)

5760

In [12]:
grid = vw.create(
    """attention head grid: 6 layers as rows × 8 heads as columns; each cell a small 15×15 heatmap of w (query q as rows, key k as columns, causal lower triangle, white to dark blue).
    label each cell with the dominant pattern computed from its weights: 'prev' if most mass sits on k = q-1, 'bos' if on k = 0, 'induction' if on the token after the previous occurrence of the same token (use input toks for the token strings), else 'mixed'; colour the label chip by pattern.
    a slider on top 'mass on previous token ≥', draggable 0–1, start 0: cells below it fade. click a cell to select it (thick orange outline); output head = {layer, head}; start with layer 1 head 3 selected.""",
    attn, inputs=vw.inputs(toks=toks), theme="minimal",
    outputs=vw.outputs(head="{layer, head} of the selected cell"),
)

In [13]:
detail = vw.create(
    """attention detail for one head, no controls of its own except a threshold slider. input head = {layer, head} selects rows of the data (layer, head, q, k, w); tokens from input toks (pos, token).
    left: the 15 tokens written in a row twice, queries on the top line and keys on the bottom line, arcs from each query down to its keys with stroke width and opacity by w; hover a query token to isolate its arcs and print its top-3 keys with weights. a slider 'hide arcs below' 0–0.5, start 0.05.
    right: the full 15×15 matrix for the same head, hover-synced with the arcs. title 'layer L · head H · <pattern>'. re-render when head changes.""",
    attn, inputs=vw.inputs(toks=toks, head=grid.outputs.head), theme="minimal",
)

### spikes · is the response locked to the stimulus
synthetic: 60 neurons over 20 s, five stimulus onsets; a third of the neurons respond ~80 ms after onset. the pair: raster picks a window and neurons, the histogram follows.

In [14]:
onsets = np.array([2.5, 6.0, 9.5, 13.0, 16.5])
spikes = []
for nrn in range(60):
    base = rng.uniform(1, 6)
    ts = rng.uniform(0, 20, rng.poisson(base * 20))
    if nrn % 3 == 0:
        for o in onsets: ts = np.r_[ts, o + 0.08 + rng.gamma(2, 0.03, rng.poisson(9))]
    spikes += [(nrn, float(x)) for x in ts]
spk = pd.DataFrame(spikes, columns=["neuron", "t"])
stim = pd.DataFrame({"onset": onsets})
len(spk)

5287

In [15]:
raster = vw.create(
    """spike raster: 60 neurons on y (0 at top), time 0–20 s on x, one 1px tick per spike (data columns neuron, t), drawn on canvas; stimulus onsets from input stim as dashed vertical lines with a small flag.
    a time window: drag on the time axis to create it (start 2.2–3.2 s), drag its body to move, edges to resize, left/right arrows shift 100 ms; the window shades the raster. the raster ticks inside the window are drawn darker.
    click a neuron's row label to toggle it into the selected set (rows highlight); shift+click a label selects the range from the last click; 'all' / 'none' buttons. start with every third neuron selected (0, 3, 6, ...).
    output window = [t0, t1] seconds, neurons = list of selected neuron ids.""",
    spk, inputs=vw.inputs(stim=stim), theme="minimal",
    outputs=vw.outputs(window="[t0, t1] in seconds", neurons="list of selected neuron ids"),
)

In [16]:
psth = vw.create(
    """firing-rate histogram that follows another widget, no controls of its own. inputs: window = [t0, t1] seconds, neurons = list of ids, stim = onsets; data = spikes (neuron, t).
    x from t0 to t1, 20 ms bins; one thin grey line per selected neuron (spikes per second) and a thick line for their mean; the stimulus onsets inside the window as dashed vertical lines with a shaded 0–150 ms strip after each; a label with the mean rate inside the strips vs outside them ('locked ×2.7'). re-render whenever window or neurons change; if no neurons, print 'pick neurons on the raster'.""",
    spk, inputs=vw.inputs(stim=stim, window=raster.outputs.window, neurons=raster.outputs.neurons), theme="minimal",
)

### pareto front · pick a design under three objectives
synthetic: 2000 candidate designs scored on cost, mass and drag. the weights are a point you drag inside a triangle.

In [17]:
d = pd.DataFrame(rng.uniform(0, 1, (2000, 3)), columns=["cost", "mass", "drag"])
d["cost"] = (d.cost ** 0.5 * 100 + (1 - d.mass) * 40).round(1)             # cheap designs are heavy
d["mass"] = (d.mass * 60 + 20).round(1)
d["drag"] = ((1 - d.mass / 80) * 0.6 + d.drag * 0.2 + 0.1).round(3)        # light designs have more drag
d["design"] = [f"D{i:04d}" for i in range(len(d))]
d.head()

,cost,mass,drag,design
0,39.1,41.0,0.459,D0000
1,69.3,28.0,0.613,D0001
2,116.4,50.2,0.381,D0002
3,124.8,24.2,0.628,D0003
4,82.4,33.6,0.621,D0004


In [18]:
pareto = vw.create(
    """pareto explorer for 2000 designs with three objectives to minimise: cost, mass, drag.
    left: an equilateral triangle whose corners are the three objectives; a draggable weight point inside it (barycentric coordinates = weights, start at the centre), the weights printed as percentages next to each corner; arrow keys nudge the point toward the last-hovered corner.
    right: a scatter of cost (x) vs mass (y) with drag as dot size; the pareto-optimal designs are outlined; the top 10 designs by weighted normalised score are filled orange with rank numbers, the rest light grey; a ranked list of the top 10 under the triangle with their three values.
    everything updates live while the point is dragged. output weights = {cost, mass, drag}, top = list of the top-10 design ids.""",
    d, theme="minimal",
    outputs=vw.outputs(weights="dict of the three weights summing to 1", top="top-10 design ids in rank order"),
)

### llm consistency · where do 24 answers to the same prompt agree
synthetic: 24 generations of a short biography from one prompt, built from a template with slots the 'model' is more or less sure about. shared spans merge into one thick band; places where the generations disagree fan out. after emily reif's consistency flow.

In [19]:
def pick(opts): return rng.choice([o for o, _ in opts], p=np.array([w for _, w in opts]) / sum(w for _, w in opts))
slots = [
    lambda: "Name:", lambda: pick([("Elara", 14), ("Eleanor", 5), ("Lydia", 3), ("Liora", 2)]),
    lambda: pick([("Voss", 9), ("Finch", 4), ("Hargrave", 3), ("Brightwell", 2), ("Windrider", 2), ("Thorne", 2), ("Moonstone", 2)]),
    lambda: pick([("(1792–1863).", 6), ("(1783–1856).", 5), ("(1823–1885).", 4), ("(1801–1875).", 3), ("(1712–1778).", 2), ("(1842–1910).", 2), ("(1623–1688).", 2)]),
    lambda: pick([("Profession:", 18), ("was a pioneering", 6)]),
    lambda: pick([("Inventor", 11), ("botanist", 6), ("Engineer.", 4), ("Astronomer;", 3)]),
    lambda: pick([("and Engineer.", 9), ("and Naturalist.", 5), ("who", 5), ("and Aviator", 3), ("and Cartographer.", 2)]),
    lambda: pick([("Contribution:", 13), ("Greatest", 6), ("Discovered", 5)]),
    lambda: pick([("Developed", 8), ("revolutionized", 6), ("pioneered", 5), ("crafted", 3), ("the", 2)]),
    lambda: pick([("the Thorne Comet", 6), ("the Solstice", 6), ("the Hargrave", 5), ("the Elmsworth Herbarium", 4), ("Windrider", 3)]),
]
gens = pd.DataFrame([{"gen": g, **{f"s{i}": f() for i, f in enumerate(slots)}} for g in range(24)])
steps = [c for c in gens.columns if c.startswith("s")]
nodes = pd.concat([gens.groupby(c).size().rename("n").reset_index().rename(columns={c: "text"}).assign(step=i) for i, c in enumerate(steps)])
links = pd.concat([gens.groupby([a, b]).size().rename("n").reset_index().rename(columns={a: "source", b: "target"}).assign(step=i) for i, (a, b) in enumerate(zip(steps, steps[1:]))])
gens["text"] = gens[steps].agg(" ".join, axis=1)
print(len(nodes), "spans,", len(links), "links"); gens.text.head(3).tolist()

41 spans, 94 links


['Name: Elara Finch (1792–1863). Profession: Inventor and Engineer. Contribution: Developed the Hargrave',
 'Name: Elara Moonstone (1842–1910). Profession: botanist who Discovered Developed the Elmsworth Herbarium',
 'Name: Elara Hargrave (1823–1885). was a pioneering Engineer. and Aviator Discovered revolutionized the Thorne Comet']

In [20]:
flow = vw.create(
    """consistency flow of 24 generations from one prompt. data = nodes (step, text, n = how many generations use that span at that step); input links (step, source, target, n = generations going from source at step to target at step+1); input gens (gen, s0..s8, text).
    steps are columns left to right, each column as wide as its longest span text plus 48px of gap so no two columns ever overlap (the diagram scrolls horizontally inside its box if the total is wider than the widget); in each column the spans are stacked with the most common at the vertical centre and rarer ones fanning above and below with at least 6px between them; write each span as text with font size scaled by n (largest ~22px, smallest ~10px, monospace). between columns draw smooth ribbons from each source span to each target span with stroke width proportional to link n, light grey-blue, semi-transparent.
    hover a span: every ribbon on the path of a generation containing that span turns dark, everything else fades; tooltip 'n of 24 generations'. click a span to pin it (bold, orange); pinned filters stay until clicked again; several pins combine with AND. under the diagram list the generations that pass through all pinned spans, full text, the pinned spans highlighted; when nothing is pinned show 'hover to trace, click to pin'.
    no controls beyond that. output pinned = list of {step, text}, gen_ids = list of gen numbers passing all pins.""",
    nodes, inputs=vw.inputs(links=links, gens=gens), theme="minimal",
    outputs=vw.outputs(pinned="list of pinned {step, text}", gen_ids="generations passing every pinned span"),
)

### genome browser · the same 50 Mb, four different pictures depending on how close you are
synthetic chromosome: 50 Mb, 420 genes with exons, 3000 variants with allele frequencies, and real letters for a 30 kb window around one gene. semantic zoom: the browser changes what it draws as you zoom, and the panel next to it changes what it *is*.

In [21]:
CHR = 50_000_000
starts = np.sort(rng.integers(0, CHR - 60_000, 420)); lens = rng.integers(2_000, 60_000, 420)
genes = pd.DataFrame({"gene": [f"G{i:03d}" for i in range(420)], "start": starts, "end": starts + lens, "strand": rng.choice(["+", "-"], 420)})
genes.loc[200, ["gene", "start", "end"]] = ["BRC1", 25_000_000, 25_024_000]         # the gene we care about
ex = []
for r in genes.itertuples():
    cuts = np.sort(rng.integers(r.start, r.end, rng.integers(2, 12) * 2))
    ex += [(r.gene, int(a), int(b)) for a, b in zip(cuts[::2], cuts[1::2])]
exons = pd.DataFrame(ex, columns=["gene", "start", "end"])
var = pd.DataFrame({"pos": np.sort(rng.integers(0, CHR, 3000))})
var["af"] = np.round(rng.beta(0.3, 3, 3000), 4); var["ref"] = rng.choice(list("ACGT"), 3000)
var["alt"] = [rng.choice([b for b in "ACGT" if b != r]) for r in var.ref]
hot = (var.pos > 25_000_000) & (var.pos < 25_024_000); var.loc[hot, "af"] = np.round(rng.uniform(0.2, 0.6, hot.sum()), 4)   # planted: common variants in BRC1
seq = pd.DataFrame({"start": [24_995_000], "seq": ["".join(rng.choice(list("ACGT"), 30_000))]})
print(len(genes), "genes", len(exons), "exons", len(var), "variants,", hot.sum(), "inside BRC1")

420 genes 2697 exons 3000 variants, 2 inside BRC1


In [22]:
browser = vw.create(
    """genome browser for one 50,000,000 bp chromosome, ~420px tall, drawn on canvas. data = variants (pos, af, ref, alt); inputs genes (gene, start, end, strand), exons (gene, start, end), seq (start, seq: the letters for one 30 kb window).
    wheel over the track zooms about the cursor, drag pans, a ruler on top shows the view in Mb/kb/bp; start with the whole chromosome in view. left/right arrows pan a tenth, +/- zoom.
    semantic zoom, the representation changes with the width of the view: over 5 Mb: a gene-density heat strip (genes per 100 kb) and a variant-density strip beneath it, no individual marks. 100 kb to 5 Mb: genes as arrows (strand direction) with names when they fit, variants as thin ticks. under 100 kb: each gene as exon boxes on a thin intron line with the name, variants as lollipops with height = af and the ref>alt written when there is room. under 2 kb: the letters of seq coloured A green C blue G orange T red, monospace, with variants highlighted; outside the seq window print 'sequence not loaded here'.
    a small level badge top-right says which of the four levels is drawn. double-click a gene at any level to zoom to it. output view = [start, end] in bp, level = 1..4.""",
    var, inputs=vw.inputs(genes=genes, exons=exons, seq=seq), theme="minimal",
    outputs=vw.outputs(view="[start, end] in bp", level="1 chromosome, 2 genes, 3 exons, 4 bases"),
)

In [23]:
inview = vw.create(
    """a panel that follows a genome browser, no controls of its own. inputs view = [start, end] bp and level = 1..4; data = variants (pos, af, ref, alt); input genes (gene, start, end, strand).
    what it draws depends on level: level 1: a histogram of allele frequency for every variant in view plus 'n variants · g genes in view'. level 2: one row per gene in view (name, length, variants inside, mean af) sorted by variants inside, at most 40 rows. level 3: one row per variant in view (pos, ref>alt, af, which gene or 'intergenic') sorted by position, af as a small bar. level 4: the variants in view written as ref>alt at their position with af, plus the count of bases in view.
    a header always shows the view span in human units (e.g. '25.00–25.02 Mb') and the level name. re-render whenever view or level changes.""",
    var, inputs=vw.inputs(genes=genes, view=browser.outputs.view, level=browser.outputs.level), theme="minimal",
)

## the last three run on opus
a live fluid solve, an ensemble map with five encodings, and a three.js scene. gemini flash produces code that runs but drops parts of the spec at this length, so these switch models.


In [24]:
vw.config(model="anthropic/claude-opus-5")

Config(provider='openrouter', host='openrouter.ai', model='anthropic/claude-opus-5', key_source='OPENROUTER_API_KEY', environment='vscode-like', temperature=0.7, timeout=120.0, streaming=True, data_privacy='sample', sample_rows=3, mode='standard', theme=None, execution='auto', retry=2, agent_preset='project', agent_run=None, bypass_row_guard=True)

### airflow · where does the air go when you move the sofa
interior design question nobody can answer by eye: the vent blows, the bookshelf blocks, a corner goes stale. a small 2d solver runs live in the widget; every piece of furniture you drag is an obstacle in it. after the Eddy3D indoor-CFD figures, without the CFD.

In [25]:
furniture = pd.DataFrame([
    ("bookshelf", 0.6, 1.2, 0.4, 2.2), ("sofa", 1.6, 3.6, 2.0, 0.9), ("desk", 5.4, 0.7, 1.6, 0.8),
    ("bed", 5.6, 3.2, 2.0, 1.6), ("plant", 3.9, 2.4, 0.5, 0.5), ("cabinet", 7.4, 1.6, 0.5, 1.4),
], columns=["name", "x", "y", "w", "h"])          # metres, x to the right, y down, (x, y) is the top-left corner
furniture

,name,x,y,w,h
0,bookshelf,0.6,1.2,0.4,2.2
1,sofa,1.6,3.6,2.0,0.9
2,desk,5.4,0.7,1.6,0.8
3,bed,5.6,3.2,2.0,1.6
4,plant,3.9,2.4,0.5,0.5
5,cabinet,7.4,1.6,0.5,1.4


In [33]:
# Inspired from De Simone, Z., Kastner, P., & Dogan, T. (2021, September). Towards safer work environments during the COVID-19 crisis: A study of different floor plan layouts and ventilation strategies coupling open FOAM and airborne pathogen data for actionable, simulation-based feedback in design. In Building Simulation 2021 (Vol. 17, pp. 2634-2641). IBPSA.
room = vw.create(
    """2d airflow sandbox for a room, top view, canvas ~600px tall. the room is 8 m wide × 5 m deep. a supply vent (blue, 0.6 m wide) on the left wall and an exhaust (grey, 0.6 m) on the right wall, both draggable along their wall.
    furniture from data (name, x, y, w, h in metres) as rounded rectangles with their name: drag to move, click a small corner handle to rotate 90°; they are solid obstacles and may not leave the room.
    run an incompressible 2d fluid solver continuously at ~30 fps on a 96×60 grid: semi-lagrangian advection, a diffusion step, pressure projection with ~24 gauss-seidel iterations, no-slip at walls and furniture, inflow at the vent with speed from a 'fan' slider (0.2–1.5 m/s, start 0.8), outflow at the exhaust. restart the solve smoothly when an obstacle moves (do not reset velocities to zero, just re-mask).
    draw speed as a viridis heatmap under everything, ~400 tracer particles advected with the flow (respawn at the vent), and hatched 'stale' cells where speed is below a draggable threshold line on a small colour legend (start 0.05 m/s). badge top-left: 'stale air: 23 % of floor'.
    every time a piece of furniture is dropped, record a snapshot in a strip on the right: a 120px thumbnail of the plan with its stale %, newest first, the lowest % marked with a star; click a thumbnail to restore that layout. keep at most 8.
    outputs arrangement = list of {name, x, y, w, h} for the current furniture positions, stale_pct = number.""",
    furniture, theme="minimal",
    outputs=vw.outputs(arrangement="current furniture rectangles", stale_pct="percent of floor cells below the stale threshold"),
)

### hurricane track · the same 200 futures, five pictures
an emergency manager has one question: does this storm hit my town. the forecast is an ensemble of 200 possible tracks; how you draw them changes what people believe. synthetic ensemble over the gulf with a real split (most members turn north, a third keep west). the town marker and its probability stay the same while you flip encodings; that is the point.

In [27]:
def track(member):
    lon, lat, hdg = -83.5, 23.5, np.deg2rad(155 + rng.normal(0, 4))     # start in the southern gulf, heading WNW (angle from east, counter-clockwise)
    turn_north = rng.random() < 0.68; turn_t = rng.uniform(25, 65)
    pts = []
    for h in range(0, 125, 5):
        if turn_north and h > turn_t: hdg -= np.deg2rad(rng.normal(3.5, 1.8))          # curve right, toward louisiana
        else: hdg += np.deg2rad(rng.normal(0.2, 1.0))
        speed = 0.55 + 0.05 * rng.normal()                                                # degrees per 5 h, ~10° in 120 h
        lon += speed * np.cos(hdg); lat += speed * np.sin(hdg) * 0.9
        pts.append((member, h, round(lon, 3), round(lat, 3)))
    return pts
tracks = pd.DataFrame([p for m in range(200) for p in track(m)], columns=["member", "t_h", "lon", "lat"])
towns = pd.DataFrame([("Houston", -95.37, 29.76), ("Corpus Christi", -97.40, 27.80), ("New Orleans", -90.07, 29.95), ("Mobile", -88.04, 30.69), ("Tampa", -82.46, 27.95)], columns=["town", "lon", "lat"])
print(len(tracks), "positions"); tracks.groupby("t_h").lon.std().round(2).tolist()[::6]

5000 positions


[0.05, 0.17, 0.37, 0.87, 1.83]

In [28]:
cone = vw.create(
    """forecast-uncertainty explorer on a leaflet map with osm tiles, ~600px tall, centred on the gulf of mexico (about 27°N, -90°E, zoom 5). data = ensemble tracks (member, t_h hours 0..120 in steps of 5, lon, lat), 200 members. input towns (town, lon, lat) drawn as small labelled pins.
    an encoding switch with five options, all computed from the same members: (1) 'cone': the mean track with a filled cone whose radius at each time is the 67th percentile of member distance from the mean, the classic look; (2) 'spaghetti': every member as a thin line; (3) 'density': member positions across all times as a heat layer (hexbin or gaussian blur on canvas); (4) 'hops': one randomly chosen member drawn boldly at a time, swapping every 500 ms, the rest invisible; (5) 'bands': the 50 % and 90 % quantile bands of position per time step, computed separately for members that end north of 28°N and those that do not, so a split shows as two bands.
    a time slider 0–120 h: the members' positions at that hour as dots plus a marker on the mean track; a small play button.
    a draggable 'my town' marker (start on Houston) with a draggable radius ring (start 80 km): a panel shows 'P(track passes within 80 km at any time) = 31 %', and 'members inside the cone at 96 h: 64 %'. a strip chart under the map: probability of having passed within the ring by hour t.
    outputs encoding, town = {lon, lat}, radius_km, p_hit.""",
    tracks, inputs=vw.inputs(towns=towns), theme="minimal",
    outputs=vw.outputs(encoding="one of cone, spaghetti, density, hops, bands", town="{lon, lat} of the marker", radius_km="ring radius", p_hit="probability any member passes within the ring"),
)

### robot grasping · where do the high-scoring grasps actually fail
synthetic: a mug as a point cloud and 140 candidate grasps with a predicted score and the outcome a simulator reported. the planted lesson: the model loves the handle, and handle grasps slip.

In [29]:
th = rng.uniform(0, 2 * np.pi, 900); z = rng.uniform(0, 0.10, 900)
body = np.c_[0.04 * np.cos(th), 0.04 * np.sin(th), z]
ht = rng.uniform(-np.pi / 2, np.pi / 2, 300)
handle = np.c_[0.04 + 0.025 * np.cos(ht) + 0.004 * rng.normal(size=300), 0.004 * rng.normal(size=300), 0.05 + 0.025 * np.sin(ht)]
cloud = pd.DataFrame(np.r_[body, handle], columns=["x", "y", "z"]).round(4)
g = []
for i in range(140):
    kind = rng.choice(["rim", "body", "handle", "base"], p=[.3, .3, .25, .15]); a = rng.uniform(0, 2 * np.pi)
    if kind == "rim":    p, ap, cl, w, score, out = (0.04 * np.cos(a), 0.04 * np.sin(a), 0.10), (0, 0, -1), (np.cos(a), np.sin(a), 0), 0.012, rng.uniform(.55, .9), rng.choice(["success", "slip"], p=[.85, .15])
    if kind == "body":   p, ap, cl, w, score, out = (0, 0, rng.uniform(.03, .08)), (-np.cos(a), -np.sin(a), 0), (-np.sin(a), np.cos(a), 0), 0.09, rng.uniform(.3, .7), rng.choice(["success", "collision"], p=[.6, .4])
    if kind == "handle": p, ap, cl, w, score, out = (0.065 + 0.006 * rng.normal(), 0, 0.05 + 0.015 * rng.normal()), (-1, 0, 0), (0, 1, 0), 0.016, rng.uniform(.8, .99), rng.choice(["slip", "success"], p=[.8, .2])
    if kind == "base":   p, ap, cl, w, score, out = (0, 0, rng.uniform(0, .02)), (-np.cos(a), -np.sin(a), 0), (-np.sin(a), np.cos(a), 0), 0.09, rng.uniform(.4, .8), rng.choice(["collision", "unreachable"], p=[.6, .4])
    g.append(dict(grasp=i, region=kind, x=p[0], y=p[1], z=p[2], ax=ap[0], ay=ap[1], az=ap[2], cx=cl[0], cy=cl[1], cz=cl[2], width=w, score=round(score, 3), outcome=out))
grasps = pd.DataFrame(g).round(4)      # (x,y,z) = point between the fingertips; (ax,ay,az) = approach direction into the object; (cx,cy,cz) = the axis the fingers close along; width = finger gap
grasps.groupby(["region", "outcome"]).size().unstack(fill_value=0)

outcome,collision,slip,success,unreachable
region,,,,
base,11,0,0,5
body,16,0,26,0
handle,0,29,8,0
rim,0,6,39,0


In [ ]:
grasp3d = vw.create(
    """3d grasp triage with three.js@0.160 and OrbitControls, ~560px tall, z up, camera looking slightly down at the object from 35 cm. a light table plane at z=0 and the object as a point cloud from input cloud (x, y, z in metres, small dark points), plus a faint translucent cylinder of radius 4 cm and height 10 cm so the mug reads as a solid.
    each row of data is a candidate parallel-jaw grasp, drawn as a U-shaped gripper glyph exactly like this: the point (x, y, z) is the midpoint between the two fingertips; the fingers are two segments 2.5 cm long, parallel to the approach direction (ax, ay, az), each ending at a fingertip; the fingertips are 'width' apart along the closing axis (cx, cy, cz), centred on (x, y, z); a palm bar joins the two finger roots. so a rim grasp looks like a small clip hanging over the rim from above and a body grasp looks like a wide U around the cylinder from the side. colour by outcome: success green, slip orange, collision red, unreachable grey; line width 2.
    side panel: 'predicted score ≥' slider (start 0.5) hiding grasps below it with 'showing k of 140'; one toggle chip per outcome with counts; a 'colour by' switch between outcome and score (viridis).
    click a glyph: it thickens, a dashed approach line 8 cm long is drawn along -approach ending at the palm, the two fingertip contact points are marked, and the panel prints id, region, score, width, outcome; the camera does not move. up/down arrows step through visible grasps in score order. click empty space to deselect.
    outputs selected = grasp id or null, min_score = slider value.""",
    grasps, inputs=vw.inputs(cloud=cloud), theme="minimal",
    outputs=vw.outputs(selected="selected grasp id or null", min_score="score slider value"),
)